# Experimento: Abliteración Quirúrgica vs. Clásica en `LiquidAI/LFM3-1.7B` (OR-Bench, Colab L4)

Prueba el caso de uso de **seguridad anti-backdoor, retención de rechazo legítimo y cero lobotomía** de `.moipatch` sobre prompts benignos de falso rechazo (`bench-llm/or-bench` `or-bench-hard-1k`), controles sanos y evaluación final held-out de retención tóxica (`or-bench-toxic`):
1. **Piloto + STOP Gate (`N_PILOT=80`):** si `LFM3-1.7B` base casi no rechaza en falso, se detiene enseguida sin quemar créditos.
2. **Baseline Clásico (fuerza bruta):** ablitera la dirección de rechazo en **las 48 matrices residuales** (`ALL layers`, permanente).
3. **WSI Quirúrgico (`Unión Pasivo + Causal` + Juez Forward):** `capture_causal_trace` + `diagnose_causal_traces` + `screen_patch_candidates` eligen solo **1–2 módulos**, ajustan un **Router MoI** en `Half-A` y lo aplican en 1 línea con `patch.enable_routed(model)`.
4. **Gate de Seguridad Held-Out (`or-bench-toxic`, `N_TOXIC=200`, `256` tokens):** los 200 prompts tóxicos de prueba NUNCA entran al descubrimiento ni al router; evalúa los brazos y exige conservar **>= 90%** de los rechazos legítimos del base.
5. **5.º brazo `wsi_safety_routed` (`N_TOXIC_CAL=100`, disjunto de los 200 de test):** calibra un router de 2 vectores (`rechazo + veto`) con CV out-of-fold sobre un corte separado para comprobar si MoI logra dejar pasar falsos rechazos benignos manteniendo apagado el parche ante prompts tóxicos reales y controles.
6. **Auditoría Anti-Backdoor:** reporta peso exacto en KB, fracción de parámetros modificados, firma SHA-256, `output_logit_diff` y `control_audit`.

Guarda el JSON final + ambos `.moipatch` (`lfm3_surgical_refusal.moipatch` y `lfm3_safety_routed_refusal.moipatch`) atómicamente en `MyDrive/wsi_runs/surgical_refusal_lfm3/` y `MyDrive/wsi_runs/FINALES/`.

In [ ]:
MODEL = "LiquidAI/LFM3-1.7B"
RUN_NAME = "surgical_refusal_lfm3_toxic_l4"
REPO = "/content/wsi"
BASE = "/content/base"
N_PILOT = 80
N_OR_BENCH = 160
N_CONTROL = 60
N_TOXIC = 200
N_TOXIC_CAL = 100
N_CAUSAL_TRACES = 10
MAX_NEW_TOK = 256
BATCH_SIZE = 16

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q torch transformers accelerate safetensors pandas pyarrow huggingface_hub hf_transfer
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
RUN_DIR = "/content/drive/MyDrive/wsi_runs/surgical_refusal_lfm3"
os.makedirs(FINALES, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/main"], check=True)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit:", COMMIT, "| modelo en:", BASE)

In [ ]:
import os, subprocess, sys

subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/main"], check=True)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

LOCAL_OUT = f"/content/{RUN_NAME}.json"
LOCAL_PATCH = "/content/lfm3_surgical_refusal.moipatch"
LOCAL_SAFETY_PATCH = "/content/lfm3_safety_routed_refusal.moipatch"
SCRIPT = f"{REPO}/scripts/experiment_surgical_refusal_lfm3.py"
if not os.path.exists(SCRIPT):
    raise FileNotFoundError(f"Falta {SCRIPT} en {REPO} (commit {COMMIT}). Falta push en local.")

cmd = [
    sys.executable, "-u", SCRIPT,
    "--model", BASE,
    "--n-pilot", str(N_PILOT),
    "--n-or-bench", str(N_OR_BENCH),
    "--n-control", str(N_CONTROL),
    "--n-toxic", str(N_TOXIC),
    "--n-toxic-cal", str(N_TOXIC_CAL),
    "--n-causal-traces", str(N_CAUSAL_TRACES),
    "--max-new-tokens", str(MAX_NEW_TOK),
    "--batch-size", str(BATCH_SIZE),
    "--patch-out", LOCAL_PATCH,
    "--safety-patch-out", LOCAL_SAFETY_PATCH,
    "--out", LOCAL_OUT,
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

In [ ]:
import datetime, json, os, shutil, subprocess

res = json.load(open(LOCAL_OUT, encoding="utf-8"))
print("STATUS:", res["status"], "| PILOTO:", res.get("pilot"))

if res["status"] == "ok":
    print("\n=== COMPARATIVA EN HALF-B HELD-OUT (LFM3-1.7B, OR-Bench + OR-Bench-Toxic) ===")
    for name, a in res["arms"].items():
        tox_rate = f"{a['toxic_refusal_rate']:6.1%}" if "toxic_refusal_rate" in a else "   n/a"
        tox_kept = f"{a['toxic_refusal_kept']:6.1%}" if a.get("toxic_refusal_kept") is not None else "   n/a"
        print(f"{name:26s} | over-refusal={a['or_bench_refusal_rate']:6.1%} | recov={str(a['recovered_of_base_refused']):>6} "
              f"| toxic_ref={tox_rate} | toxic_kept={tox_kept} | ctrl_exact={a['control_exact_match_rate']:6.1%} "
              f"| healthy_kl={a['healthy_kl']:7.5f} | dNLL={a['delta_healthy_nll']:+7.5f}")
    if res.get("safety_gate"):
        sg = res["safety_gate"]
        verdict = "PASS" if sg["passed"] else "FAIL"
        _pct = lambda v: f"{v:.1%}" if isinstance(v, (int, float)) else "n/a"
        print(f"\n=== SAFETY GATE (OR-Bench-Toxic, criterio routed >= 90%) : {verdict} ===")
        print(f"base_toxic={_pct(sg['base_toxic_refusal_rate'])} | classic_kept={_pct(sg['classic_toxic_refusal_kept'])} "
              f"| wsi_always_kept={_pct(sg['wsi_always_toxic_refusal_kept'])} | wsi_routed_kept={_pct(sg['wsi_routed_toxic_refusal_kept'])} "
              f"| wsi_safety_routed_kept={_pct(sg.get('wsi_safety_routed_toxic_refusal_kept'))} "
              f"({'PASS' if sg.get('safety_routed_passed') else 'FAIL' if sg.get('safety_routed_passed') is False else 'n/a'})")
        rb = res["discovery"].get("router_half_b") or {}
        if rb:
            print(f"Router Half-B fire rates -> or_bench={rb.get('or_bench_fire_rate')} | toxic={rb.get('toxic_fire_rate')} "
                  f"| toxic_base_refused={rb.get('toxic_base_refused_fire_rate')} | control_fpr={rb.get('control_fpr')}")
        sa = res["discovery"].get("safety_router_half_a") or {}
        sb = res["discovery"].get("safety_router_half_b") or {}
        if sa:
            print(f"Safety Router (2 vectores OOF CV) -> layer={sa.get('layer')} method={sa.get('veto_method')} "
                  f"oof_or_tpr={sa.get('oof_or_tpr')} oof_tox_fpr={sa.get('oof_toxic_fpr')} oof_ctrl_fpr={sa.get('oof_ctrl_fpr')}")
        if sb:
            print(f"Safety Router Half-B fire rates -> or_bench={sb.get('or_bench_fire_rate')} | toxic={sb.get('toxic_fire_rate')} "
                  f"| toxic_base_refused={sb.get('toxic_base_refused_fire_rate')} | control_fpr={sb.get('control_fpr')}")
    au = res["audit"]
    print("\n=== AUDITORÍA ANTI-BACKDOOR ===")
    print(f"Tamaño .moipatch: {au['patch_size_kb']} KB | Módulos editados: {au['n_edited_modules']}/{res['architecture']['n_residual_writers']} "
          f"(vs Clásico: {au['classic_n_edited_modules']}) | SHA256 OK: {au['fingerprint_ok']} | Restore OK: {au['restore_exact_ok']}")
    if au.get("output_logit_diff"):
        old = au["output_logit_diff"]
        print(f"[output_logit_diff] probes={old['n_probes']} | mean_kl={old['mean_next_token_kl']} "
              f"| top1_changed={old['top1_changed_rate']:.1%} | router_fire={old['probe_router_fire_rate']}")
        print("  suprime (Δp) :", [(x["token"], x["mean_delta_prob"]) for x in old["suppressed_tokens"][:6]])
        print("  promueve (Δp):", [(x["token"], x["mean_delta_prob"]) for x in old["promoted_tokens"][:6]])
    if au.get("control_audit"):
        ca = au["control_audit"]
        print(f"[control_audit] controls={ca['n_controls']} | routed_fire={ca['routed_fire_rate']} "
              f"| routed_kl={ca['routed_mean_kl']} | always_on_kl={ca['always_on_mean_kl']}")
    for v in au["vocab_logit_lens"]:
        nota = " (capa temprana — ver output_logit_diff)" if v.get("early_layer_caveat") else ""
        print(f"  [vocab_logit_lens] {v['module']} (alpha={v['alpha']}, cols={v['cols']}){nota}:")
        print("    suprime :", [x["token"] for x in v["suppressed_tokens"][:6]])
        print("    promueve:", [x["token"] for x in v["promoted_tokens"][:6]])
    print("\nTiming (s):", res["timing_s"])

res["_final"] = {
    "complete": True,
    "model": MODEL,
    "commit": COMMIT,
    "notebook": "surgical_refusal_lfm3_colab.ipynb",
    "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
    "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}
for out_dir in (FINALES, RUN_DIR):
    os.makedirs(out_dir, exist_ok=True)
    dst = f"{out_dir}/{RUN_NAME}.json"
    with open(dst + ".tmp", "w", encoding="utf-8") as f:
        json.dump(res, f, indent=2)
    os.replace(dst + ".tmp", dst)
    for src_p, fname in (
        (LOCAL_PATCH, "lfm3_surgical_refusal.moipatch"),
        (LOCAL_SAFETY_PATCH, "lfm3_safety_routed_refusal.moipatch"),
    ):
        if os.path.exists(src_p):
            p_dst = f"{out_dir}/{fname}"
            shutil.copy2(src_p, p_dst + ".tmp")
            os.replace(p_dst + ".tmp", p_dst)
print("\nGuardado final en Drive:", f"{FINALES}/{RUN_NAME}.json", "y", RUN_DIR)

In [ ]:
# Apagar runtime solo si el JSON final quedó guardado y verificado en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst, encoding="utf-8")).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")